In [0]:
print("Hello World")
print("I have also joined the group & this is my homework")

Hello World
I have also joined the group & this is my homework


In [0]:
display(spark.sql("DESCRIBE adventurework.silver.fact_sales"))

col_name,data_type,comment
SalesOrderID,int,null
SalesOrderDetailID,int,null
CarrierTrackingNumber,string,null
OrderQty,smallint,null
ProductID,int,null
SpecialOfferID,int,null
UnitPrice,"decimal(19,4)",null
UnitPriceDiscount,"decimal(19,4)",null
LineTotal,"decimal(19,4)",null
rowguid,string,null


In [0]:

 %sql SELECT COUNT(*) FROM adventurework.silver.fact_sales, adventurework.silver.dim_product, adventurework.silver.dim_sales_territory

COUNT(*)
611437680


In [0]:
 %sql SELECT COUNT(*) FROM adventurework.silver.fact_sales

COUNT(*)
121317


In [0]:
 %sql SELECT COUNT(*) FROM adventurework.silver.dim_product

COUNT(*)
504


In [0]:
%sql SELECT COUNT(*) FROM adventurework.silver.dim_sales_territory

COUNT(*)
10


In [0]:
%sql SELECT COUNT(Distinct SalesOrderDetailID) FROM adventurework.silver.fact_sales
--, adventurework.silver.dim_product, adventurework.silver.dim_sales_territory

COUNT(DISTINCTSalesOrderDetailID)
121317


In [0]:
%sql 
--select * from adventurework.silver.dim_sales_territory limit 5
SELECT COUNT(Distinct TerritoryID) FROM adventurework.silver.dim_sales_territory
--, adventurework.silver.dim_product, adventurework.silver.dim_sales_territory

COUNT(DISTINCTTerritoryID)
10


In [0]:
%sql
-- LEFT JOIN: keeps ALL rows from fact_sales, even if there's no match in the dimension
SELECT 
    f.SalesOrderID,
    f.SalesOrderDetailID,
    f.ProductID,
    p.Name AS ProductName,
    p.Color,
    p.ListPrice
FROM adventurework.silver.fact_sales f
LEFT JOIN adventurework.silver.dim_product p
    ON f.ProductID = p.ProductID
LIMIT 10

SalesOrderID,SalesOrderDetailID,ProductID,ProductName,Color,ListPrice
43659,1,776,"Mountain-100 Black, 42",Black,3374.9900
43659,2,777,"Mountain-100 Black, 44",Black,3374.9900
43659,3,778,"Mountain-100 Black, 48",Black,3374.9900
43659,4,771,"Mountain-100 Silver, 38",Silver,3399.9900
43659,5,772,"Mountain-100 Silver, 42",Silver,3399.9900
43659,6,773,"Mountain-100 Silver, 44",Silver,3399.9900
43659,7,774,"Mountain-100 Silver, 48",Silver,3399.9900
43659,8,714,"Long-Sleeve Logo Jersey, M",Multi,49.9900
43659,9,716,"Long-Sleeve Logo Jersey, XL",Multi,49.9900
43659,10,709,"Mountain Bike Socks, M",White,9.5000


In [0]:
%sql
-- Count total rows in the LEFT JOIN result
SELECT COUNT(*) AS total_rows
FROM adventurework.silver.fact_sales f
LEFT JOIN adventurework.silver.dim_product p
    ON f.ProductID = p.ProductID

total_rows
121317


In [0]:
%sql
-- RIGHT JOIN: keeps ALL rows from dim_product (the right table), even if no match in fact_sales
SELECT 
    f.SalesOrderID,
    f.SalesOrderDetailID,
    f.ProductID,
    p.Name AS ProductName,
    p.Color,
    p.ListPrice
FROM adventurework.silver.fact_sales f
RIGHT JOIN adventurework.silver.dim_product p
    ON f.ProductID = p.ProductID
LIMIT 10

SalesOrderID,SalesOrderDetailID,ProductID,ProductName,Color,ListPrice
null,null,null,Adjustable Race,null,0.0000
null,null,null,Bearing Ball,null,0.0000
null,null,null,BB Ball Bearing,null,0.0000
null,null,null,Headset Ball Bearings,null,0.0000
null,null,null,Blade,null,0.0000
null,null,null,LL Crankarm,Black,0.0000
null,null,null,ML Crankarm,Black,0.0000
null,null,null,HL Crankarm,Black,0.0000
null,null,null,Chainring Bolts,Silver,0.0000
null,null,null,Chainring Nut,Silver,0.0000


In [0]:
%sql
-- Count total rows in the RIGHT JOIN result
SELECT COUNT(*) AS total_rows
FROM adventurework.silver.fact_sales f
RIGHT JOIN adventurework.silver.dim_product p
    ON f.ProductID = p.ProductID

total_rows
121555


In [0]:
%sql
-- Find ProductID values that appear more than once in fact_sales (i.e. sold multiple times)
SELECT 
    f.ProductID,
    p.Name AS ProductName,
    COUNT(*) AS row_count
FROM adventurework.silver.fact_sales f
JOIN adventurework.silver.dim_product p
    ON f.ProductID = p.ProductID
GROUP BY f.ProductID, p.Name
HAVING COUNT(*) > 1
ORDER BY row_count DESC
LIMIT 15

ProductID,ProductName,row_count
870,Water Bottle - 30 oz.,4688
712,AWC Logo Cap,3382
873,Patch Kit/8 Patches,3354
921,Mountain Tire Tube,3095
711,"Sport-100 Helmet, Blue",3090
707,"Sport-100 Helmet, Red",3083
708,"Sport-100 Helmet, Black",3007
922,Road Tire Tube,2376
878,Fender Set - Mountain,2121
871,Mountain Bottle Cage,2025


In [0]:
%sql
-- DEMO: How a duplicate ProductID in a dimension table silently doubles revenue

-- Step 1: Check dim_product has NO duplicate ProductIDs (correct behavior)
SELECT ProductID, COUNT(*) AS dup_count
FROM adventurework.silver.dim_product
GROUP BY ProductID
HAVING COUNT(*) > 1;
-- Expected: 0 rows (no duplicates) -- this is what a dimension table SHOULD look like

-- Step 2: Simulate what happens IF dim_product had a duplicate ProductID
-- We artificially duplicate ProductID = 776 (Mountain-100 Black, 42)
WITH dim_product_duplicated AS (
    SELECT * FROM adventurework.silver.dim_product
    WHERE ProductID = 776
    UNION ALL
    SELECT * FROM adventurework.silver.dim_product
    WHERE ProductID = 776  -- same row added a second time (simulating a duplicate key)
),
fact_sales_filtered AS (
    SELECT * FROM adventurework.silver.fact_sales
    WHERE ProductID = 776
)
-- Join fact to the DUPLICATED dimension
SELECT 
    f.SalesOrderID,
    f.ProductID,
    f.LineTotal AS revenue_per_line,
    p.Name AS ProductName
FROM fact_sales_filtered f
LEFT JOIN dim_product_duplicated p
    ON f.ProductID = p.ProductID
ORDER BY f.SalesOrderID, f.SalesOrderDetailID
LIMIT 15;
-- Notice: each fact row now appears TWICE because it matches BOTH dimension rows for ProductID 776

-- Step 3: Show the revenue impact side-by-side
WITH dim_product_clean AS (
    SELECT * FROM adventurework.silver.dim_product WHERE ProductID = 776
),
dim_product_duped AS (
    SELECT * FROM adventurework.silver.dim_product WHERE ProductID = 776
    UNION ALL
    SELECT * FROM adventurework.silver.dim_product WHERE ProductID = 776
),
fact_filtered AS (
    SELECT * FROM adventurework.silver.fact_sales WHERE ProductID = 776
)
SELECT 
    'Clean (no dupes)' AS scenario,
    SUM(f.LineTotal) AS total_revenue,
    COUNT(*) AS row_count
FROM fact_filtered f
LEFT JOIN dim_product_clean p ON f.ProductID = p.ProductID
UNION ALL
SELECT 
    'Duplicated key' AS scenario,
    SUM(f.LineTotal) AS total_revenue,
    COUNT(*) AS row_count
FROM fact_filtered f
LEFT JOIN dim_product_duped p ON f.ProductID = p.ProductID;
-- The duplicated scenario has EXACTLY 2x the revenue and 2x the rows -- silently inflated!

scenario,total_revenue,row_count
Clean (no dupes),1254722.7326,228
Duplicated key,2509445.4652,456


In [0]:
%sql
-- Compare total revenue: ALL products with clean keys vs ALL products with one duplicated key
WITH dim_product_clean AS (
    SELECT * FROM adventurework.silver.dim_product
),
dim_product_duped AS (
    -- Duplicate ONLY ProductID = 776, keep everything else clean
    SELECT * FROM adventurework.silver.dim_product
    UNION ALL
    SELECT * FROM adventurework.silver.dim_product WHERE ProductID = 776
)
SELECT 
    'Clean (no dupes)' AS scenario,
    SUM(f.LineTotal) AS total_revenue,
    COUNT(*) AS row_count
FROM adventurework.silver.fact_sales f
LEFT JOIN dim_product_clean p ON f.ProductID = p.ProductID
UNION ALL
SELECT 
    'One duplicated key (776)' AS scenario,
    SUM(f.LineTotal) AS total_revenue,
    COUNT(*) AS row_count
FROM adventurework.silver.fact_sales f
LEFT JOIN dim_product_duped p ON f.ProductID = p.ProductID;
-- Only the revenue from ProductID 776 is doubled; everything else stays the same

scenario,total_revenue,row_count
Clean (no dupes),109846381.4250,121317
One duplicated key (776),111101104.1576,121545


In [0]:
%sql select * from adventurework.silver.fact_sales f
ORDER BY f.ModifiedDate
limit 10

SalesOrderID,SalesOrderDetailID,CarrierTrackingNumber,OrderQty,ProductID,SpecialOfferID,UnitPrice,UnitPriceDiscount,LineTotal,rowguid,ModifiedDate
43659,9,4911-403C-98,1,716,1,28.8404,0.0000,28.8404,AA542630-BDCD-4CE5-89A0-C1BF82747725,2022-05-30T00:00:00.000Z
43659,6,4911-403C-98,2,773,1,2039.9940,0.0000,4079.9880,CE472532-A4C0-45BA-816E-EEFD3FD848B3,2022-05-30T00:00:00.000Z
43659,7,4911-403C-98,1,774,1,2039.9940,0.0000,2039.9940,80667840-F962-4EE3-96E0-AECA108E0D4F,2022-05-30T00:00:00.000Z
43659,8,4911-403C-98,3,714,1,28.8404,0.0000,86.5212,E9D54907-E7B7-4969-80D9-76BA69F8A836,2022-05-30T00:00:00.000Z
43659,2,4911-403C-98,3,777,1,2024.9940,0.0000,6074.9820,7ABB600D-1E77-41BE-9FE5-B9142CFC08FA,2022-05-30T00:00:00.000Z
43659,5,4911-403C-98,1,772,1,2039.9940,0.0000,2039.9940,5A74C7D2-E641-438E-A7AC-37BF23280301,2022-05-30T00:00:00.000Z
43659,1,4911-403C-98,1,776,1,2024.9940,0.0000,2024.9940,B207C96D-D9E6-402B-8470-2CC176C42283,2022-05-30T00:00:00.000Z
43659,4,4911-403C-98,1,771,1,2039.9940,0.0000,2039.9940,04C4DE91-5815-45D6-8670-F462719FBCE3,2022-05-30T00:00:00.000Z
43659,3,4911-403C-98,1,778,1,2024.9940,0.0000,2024.9940,475CF8C6-49F6-486E-B0AD-AFC6A50CDD2F,2022-05-30T00:00:00.000Z
43659,10,4911-403C-98,6,709,1,5.7000,0.0000,34.2000,AC769034-3C2F-495C-A5A7-3B71CDB25D4E,2022-05-30T00:00:00.000Z


In [0]:
%sql
-- Data quality check: find rows with NULL or negative LineTotal
SELECT 
    'NULL LineTotal' AS issue_type,
    COUNT(*) AS row_count
FROM adventurework.silver.fact_sales
WHERE LineTotal IS NULL
UNION ALL
SELECT 
    'Negative LineTotal' AS issue_type,
    COUNT(*) AS row_count
FROM adventurework.silver.fact_sales
WHERE LineTotal < 0
UNION ALL
SELECT 
    'Zero LineTotal' AS issue_type,
    COUNT(*) AS row_count
FROM adventurework.silver.fact_sales
WHERE LineTotal = 0;

-- If any issues found, inspect the actual problem rows:
SELECT 
    SalesOrderID,
    SalesOrderDetailID,
    ProductID,
    OrderQty,
    UnitPrice,
    UnitPriceDiscount,
    LineTotal,
    ModifiedDate
FROM adventurework.silver.fact_sales
WHERE LineTotal IS NULL OR LineTotal < 0 OR LineTotal = 0
ORDER BY SalesOrderID
LIMIT 20;

issue_type,row_count
NULL LineTotal,0
Negative LineTotal,0
Zero LineTotal,0


In [0]:
%sql
-- Compare: total fact rows vs distinct ProductIDs in fact vs distinct products in dimension
SELECT 
    'Total rows in fact_sales' AS metric,
    COUNT(*) AS value
FROM adventurework.silver.fact_sales
UNION ALL
SELECT 
    'Distinct ProductIDs in fact_sales' AS metric,
    COUNT(DISTINCT ProductID) AS value
FROM adventurework.silver.fact_sales
UNION ALL
SELECT 
    'Distinct ProductIDs in dim_product' AS metric,
    COUNT(DISTINCT ProductID) AS value
FROM adventurework.silver.dim_product;

-- What this tells you:
-- 1) Total rows >> Distinct ProductIDs in fact_sales → each product is sold many times (expected)
-- 2) If Distinct ProductIDs in fact_sales < Distinct ProductIDs in dim_product → some products were never sold
-- 3) If Distinct ProductIDs in fact_sales > Distinct ProductIDs in dim_product → orphaned fact rows (no matching product!)

metric,value
Distinct ProductIDs in fact_sales,266
Distinct ProductIDs in dim_product,504
Total rows in fact_sales,121317


In [0]:
%sql
-- Top 10 products by total revenue (LineTotal)
SELECT 
    p.ProductID,
    p.Name AS ProductName,
    p.Color,
    p.ListPrice,
    COUNT(*) AS units_sold,
    SUM(f.LineTotal) AS total_revenue
FROM adventurework.silver.fact_sales f
JOIN adventurework.silver.dim_product p
    ON f.ProductID = p.ProductID
GROUP BY p.ProductID, p.Name, p.Color, p.ListPrice
ORDER BY total_revenue DESC
LIMIT 10

ProductID,ProductName,Color,ListPrice,units_sold,total_revenue
782,"Mountain-200 Black, 38",Black,2294.9900,1252,4400592.8004
783,"Mountain-200 Black, 42",Black,2294.9900,1177,4009494.7619
779,"Mountain-200 Silver, 38",Silver,2319.9900,1094,3693678.0254
780,"Mountain-200 Silver, 42",Silver,2319.9900,1040,3438478.8606
781,"Mountain-200 Silver, 46",Silver,2319.9900,1054,3434256.9420
784,"Mountain-200 Black, 46",Black,2294.9900,1059,3309673.2170
793,"Road-250 Black, 44",Black,2443.3500,705,2516857.3149
794,"Road-250 Black, 48",Black,2443.3500,712,2347655.9534
795,"Road-250 Black, 52",Black,2443.3500,667,2012447.7750
753,"Road-150 Red, 56",Red,3578.2700,475,1847818.6280


In [0]:
df_top5 = spark.sql("""
SELECT 
    p.Name AS ProductName,
    SUM(f.LineTotal) AS total_revenue
FROM adventurework.silver.fact_sales f
JOIN adventurework.silver.dim_product p
    ON f.ProductID = p.ProductID
GROUP BY p.Name
ORDER BY total_revenue DESC
LIMIT 5
""")
display(df_top5, option={"type": "bar", "key": "ProductName", "values": "total_revenue"})

ProductName,total_revenue
"Mountain-200 Black, 38",4400592.8004
"Mountain-200 Black, 42",4009494.7619
"Mountain-200 Silver, 38",3693678.0254
"Mountain-200 Silver, 42",3438478.8606
"Mountain-200 Silver, 46",3434256.9420


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT COUNT(*), COUNT(DISTINCT ProductID) 
FROM adventurework.silver.dim_product

COUNT(*),COUNT(DISTINCTProductID)
504,504


In [0]:
%sqlSELECT COUNT(*), COUNT(DISTINCT ProductID) 
FROM adventurework.silver.dim_product

In [0]:
%sql DESCRIBE adventurework.silver.dim_sales_territory

col_name,data_type,comment
TerritoryID,int,null
Name,string,null
CountryRegionCode,string,null
Group,string,null
SalesYTD,"decimal(19,4)",null
SalesLastYear,"decimal(19,4)",null
CostYTD,"decimal(19,4)",null
CostLastYear,"decimal(19,4)",null
rowguid,string,null
ModifiedDate,timestamp,null


In [0]:
%sql
SELECT COUNT(*), COUNT(DISTINCT TerritoryID) 
FROM adventurework.silver.dim_sales_territory

COUNT(*),COUNT(DISTINCTTerritoryID)
10,10


In [0]:
%sql DESCRIBE adventurework.silver.fact_sales

col_name,data_type,comment
SalesOrderID,int,null
SalesOrderDetailID,int,null
CarrierTrackingNumber,string,null
OrderQty,smallint,null
ProductID,int,null
SpecialOfferID,int,null
UnitPrice,"decimal(19,4)",null
UnitPriceDiscount,"decimal(19,4)",null
LineTotal,"decimal(19,4)",null
rowguid,string,null


In [0]:
%sql
SELECT COUNT(*), COUNT(DISTINCT SalesOrderID) 
FROM adventurework.silver.fact_sales

COUNT(*),COUNT(DISTINCTSalesOrderID)
121317,31465


In [0]:
%sql
-- CHECK 1: Orphaned fact rows — ProductIDs in fact_sales with NO match in dim_product
-- LEFT JOIN + WHERE p.ProductID IS NULL finds fact rows that reference a product that doesn't exist
SELECT 
    'Orphaned fact rows (no matching product)' AS check_name,
    COUNT(*) AS row_count
FROM adventurework.silver.fact_sales f
LEFT JOIN adventurework.silver.dim_product p
    ON f.ProductID = p.ProductID
WHERE p.ProductID IS NULL

UNION ALL

-- CHECK 2: Unsold products — Products in dim_product with NO sales in fact_sales
-- LEFT JOIN + WHERE f.ProductID IS NULL finds products that were never sold
SELECT 
    'Unsold products (no matching sales)' AS check_name,
    COUNT(*) AS row_count
FROM adventurework.silver.dim_product p
LEFT JOIN adventurework.silver.fact_sales f
    ON f.ProductID = p.ProductID
WHERE f.ProductID IS NULL;

check_name,row_count
Orphaned fact rows (no matching product),0
Unsold products (no matching sales),238


In [0]:
%sql
-- Products ordered most frequently: by number of order lines and distinct orders
SELECT 
    p.ProductID,
    p.Name AS ProductName,
    p.Color,
    COUNT(*) AS order_line_count,
    COUNT(DISTINCT f.SalesOrderID) AS distinct_orders,
    SUM(f.OrderQty) AS total_quantity_sold
FROM adventurework.silver.fact_sales f
JOIN adventurework.silver.dim_product p
    ON f.ProductID = p.ProductID
GROUP BY p.ProductID, p.Name, p.Color
ORDER BY order_line_count DESC
LIMIT 10

ProductID,ProductName,Color,order_line_count,distinct_orders,total_quantity_sold
870,Water Bottle - 30 oz.,null,4688,4688,6815
712,AWC Logo Cap,Multi,3382,3382,8311
873,Patch Kit/8 Patches,null,3354,3354,3865
921,Mountain Tire Tube,null,3095,3095,3095
711,"Sport-100 Helmet, Blue",Blue,3090,3090,6743
707,"Sport-100 Helmet, Red",Red,3083,3083,6266
708,"Sport-100 Helmet, Black",Black,3007,3007,6532
922,Road Tire Tube,null,2376,2376,2376
878,Fender Set - Mountain,null,2121,2121,2121
871,Mountain Bottle Cage,null,2025,2025,2025


In [0]:
%sql
-- Revenue by product line
SELECT 
    p.ProductLine,
    p.Name AS ProductName,
    COUNT(*) AS order_lines,
    SUM(f.OrderQty) AS total_quantity,
    SUM(f.LineTotal) AS total_revenue,
    ROUND(SUM(f.LineTotal) / (SELECT SUM(LineTotal) FROM adventurework.silver.fact_sales) * 100, 2) AS pct_of_total_revenue
FROM adventurework.silver.fact_sales f
JOIN adventurework.silver.dim_product p
    ON f.ProductID = p.ProductID
GROUP BY p.ProductLine, p.Name
ORDER BY total_revenue DESC
Limit 1

ProductLine,ProductName,order_lines,total_quantity,total_revenue,pct_of_total_revenue
M,"Mountain-200 Black, 38",1252,2977,4400592.8004,4.01
